Dodać opisy

In [ ]:
import pandas as pd
import zipfile

with zipfile.ZipFile("gtfs-2024-2025.zip", "r") as z:
    stops = pd.read_csv(z.open("stops.txt"))
    stop_times = pd.read_csv(z.open("stop_times.txt"))
    routes = pd.read_csv(z.open("routes.txt"))
    trips = pd.read_csv(z.open("trips.txt"))

df = stop_times.merge(stops, on="stop_id", how="left")
df2 = trips.merge(routes, on="route_id", how="left")

In [ ]:
import folium
import geopandas as gpd

train_stops = gpd.read_file("export.geojson")

m = folium.Map(location=[51.76, 19.46], zoom_start=9)

for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(m)

m

In [ ]:
m = folium.Map(location=[51.76, 19.46], zoom_start=9)

voivodeships = gpd.read_file("A01_Granice_wojewodztw.shp")
lodzkie = voivodeships[voivodeships['JPT_NAZWA_'] == 'łódzkie']
lodzkie_proj = lodzkie.to_crs(epsg=2180)


centroid_proj = lodzkie_proj.geometry.centroid.iloc[0]
centroid_proj = gpd.GeoSeries([centroid_proj], crs=2180).to_crs(epsg=4326).iloc[0]
lodzkie_proj = lodzkie.to_crs(epsg=4326)

folium.GeoJson(
    lodzkie_proj,
    name="Województwo łódzkie",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)



train_stops = train_stops.to_crs(epsg=4326)
train_stops = train_stops[train_stops.geometry.within(lodzkie_proj.geometry.iloc[0])]



for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(m)

m

In [ ]:
railways = gpd.read_file("tmp_shp\gis_osm_railways_free_1.shp")
roads = gpd.read_file("tmp_shp\gis_osm_roads_free_1.shp")

railways = railways.to_crs(epsg=4326)
roads = roads.to_crs(epsg=4326)



m = folium.Map(location=[51.76, 19.46], zoom_start=9)

car_fclasses = [
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
]

roads_car = roads[roads["fclass"].isin(car_fclasses)].copy()

fg_roads = folium.FeatureGroup(name="Drogi samochodowe", show=False)
folium.GeoJson(
    roads_car,
    name="Drogi",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1
    }
).add_to(fg_roads)
fg_roads.add_to(m)

tram_railways = railways[railways["fclass"] == "tram"]

fg_tram = folium.FeatureGroup(name="Linie tramwajowe", show=False)
folium.GeoJson(
    tram_railways,
    name="Tramwaje",
    style_function=lambda x: {
        "color": "brown",
        "weight": 1
    }
).add_to(fg_tram)
fg_tram.add_to(m)

train_railways = railways[railways["fclass"] == "rail"]

fg_rail = folium.FeatureGroup(name="Linie kolejowe")
folium.GeoJson(
    train_railways,
    name="Kolej",
    style_function=lambda x: {
        "color": "black",
        "weight": 2
    }
).add_to(fg_rail)
fg_rail.add_to(m)

folium.GeoJson(
    lodzkie_proj,
    control=False,
    name="Województwo łódzkie",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)

fg_stops = folium.FeatureGroup(name="Stacje kolejowe")
for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(fg_stops)
fg_stops.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
import numpy as np
import networkx as nx
from scipy.spatial import cKDTree
from shapely.geometry import Point
from matplotlib import cm as mpl_cm
from matplotlib import colors
import matplotlib.pyplot as plt

train_all_stops = gpd.read_file("export.geojson")
train_all_stops = train_all_stops.to_crs(epsg=2180)
lodzkie_proj = lodzkie.to_crs(epsg=2180)
roads = roads.to_crs(epsg=2180)
roads["length_m"] = roads.geometry.length
roads["walk_speed_mps"] = 5.0 / 3.6
roads["walk_time_s"] = roads["length_m"] / roads["walk_speed_mps"]

G = nx.Graph()

for idx, row in roads.iterrows():
    geom = row.geometry
    if geom.geom_type == "LineString":
        coords = list(geom.coords)
        for u, v in zip(coords[:-1], coords[1:]):
            G.add_edge(
                u, v,
                weight=row["walk_time_s"]
            )


nodes = np.array(G.nodes)

def nearest_node(point, nodes):
    dists = np.sqrt((nodes[:,0] - point.x)**2 + (nodes[:,1] - point.y)**2)
    return tuple(nodes[dists.argmin()])

station_nodes = [
    nearest_node(pt, nodes)
    for pt in train_all_stops.geometry
]

times = nx.multi_source_dijkstra_path_length(
    G,
    sources=station_nodes,
    weight="weight"
)


pixel_size = 1000

minx, miny, maxx, maxy = lodzkie_proj.total_bounds
width = int((maxx - minx) / pixel_size) + 1
height = int((maxy - miny) / pixel_size) + 1
x_coords = np.linspace(minx, maxx, width)
y_coords = np.linspace(miny, maxy, height)
xx, yy = np.meshgrid(x_coords, y_coords[::-1])


nodes = np.array(list(G.nodes))

times_arr = np.array([
    times.get(tuple(n), np.nan) for n in nodes
])

tree = cKDTree(nodes)


_, idxs = tree.query(np.c_[xx.ravel(), yy.ravel()])
time_raster = times_arr[idxs].reshape(height, width)

px_points = [Point(x, y) for x, y in zip(xx.ravel(), yy.ravel())]
mask = np.array([lodzkie_proj.contains(pt).any() for pt in px_points])
mask = mask.reshape(height, width)
time_raster_masked = np.where(mask, time_raster, np.nan)

finite_vals = time_raster_masked[np.isfinite(time_raster_masked)]
vmin = np.percentile(finite_vals, 2)
vmax = np.percentile(finite_vals, 25)

norm = colors.Normalize(vmin=vmin, vmax=vmax)
time_norm = norm(time_raster_masked)

cmap = mpl_cm.get_cmap("RdYlGn_r")
img = cmap(time_norm)
alpha = np.ones_like(time_raster_masked)
alpha[np.isnan(time_raster_masked) & mask] = 1.0
alpha[np.isnan(time_raster_masked) & ~mask] = 0.0
img[..., 3] = alpha
img[np.isnan(time_raster_masked) & mask, :3] = 0

plt.imsave("walk_travel_time.png", img)


car_fclasses = [
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "residential", "living_street"
]

roads_detail_car = roads[roads["fclass"].isin(car_fclasses)].copy()
roads_detail_car = roads_detail_car.to_crs(epsg=2180)
roads_detail_car["length_m"] = roads_detail_car.geometry.length
roads_detail_car["speed_mps"] = roads_detail_car["maxspeed"].mask((roads_detail_car["maxspeed"] == 0) | roads_detail_car["maxspeed"].isna(),50) / 3.6
roads_detail_car["drive_time_s"] = roads_detail_car["length_m"] / roads_detail_car["speed_mps"]
roads_detail_car["drive_time_s"] = roads_detail_car["drive_time_s"].replace([np.inf, -np.inf], np.nan)

G = nx.Graph()

for idx, row in roads_detail_car.iterrows():
    geom = row.geometry
    if np.isnan(row["drive_time_s"]) or np.isinf(row["drive_time_s"]):
        continue
    if geom.geom_type == "LineString":
        coords = list(geom.coords)
        for u, v in zip(coords[:-1], coords[1:]):
            G.add_edge(
                u, v,
                weight=row["drive_time_s"]
            )

nodes = np.array(G.nodes)

station_nodes = [
    nearest_node(pt, nodes)
    for pt in train_all_stops.geometry
]

times = nx.multi_source_dijkstra_path_length(
    G,
    sources=station_nodes,
    weight="weight"
)

nodes = np.array(list(G.nodes))

times_arr = np.array([
    times.get(tuple(n), np.nan) for n in nodes
])

tree = cKDTree(nodes)

_, idxs = tree.query(np.c_[xx.ravel(), yy.ravel()])
time_raster = times_arr[idxs].reshape(height, width)
time_raster_masked = np.where(mask, time_raster, np.nan)

finite_vals = time_raster_masked[np.isfinite(time_raster_masked)]
vmin = np.percentile(finite_vals, 5)
vmax = np.percentile(finite_vals, 50)

norm = colors.Normalize(vmin=vmin, vmax=vmax)
time_norm = norm(time_raster_masked)

cmap = mpl_cm.get_cmap("RdYlGn_r")
img = cmap(time_norm)
alpha = np.ones_like(time_raster_masked)
alpha[np.isnan(time_raster_masked) & mask] = 1.0
alpha[np.isnan(time_raster_masked) & ~mask] = 0.0
img[..., 3] = alpha
img[np.isnan(time_raster_masked) & mask, :3] = 0

plt.imsave("drive_travel_time.png", img)

In [ ]:
from pyproj import Transformer

train_stops = train_stops.to_crs(epsg=4326)
lodzkie_proj = lodzkie.to_crs(epsg=4326)
roads_car = roads_car.to_crs(epsg=4326)

transformer = Transformer.from_crs("EPSG:2180", "EPSG:4326", always_xy=True)
lon_min, lat_min = transformer.transform(minx, miny)
lon_max, lat_max = transformer.transform(maxx, maxy)

m = folium.Map(location=[51.76, 19.46], zoom_start=9)

fg_roads = folium.FeatureGroup(name="Drogi samochodowe", show=False)
folium.GeoJson(
    roads_car,
    name="Drogi",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1
    }
).add_to(fg_roads)
fg_roads.add_to(m)

fg_tram = folium.FeatureGroup(name="Linie tramwajowe", show=False)
folium.GeoJson(
    tram_railways,
    name="Tramwaje",
    style_function=lambda x: {
        "color": "brown",
        "weight": 1
    }
).add_to(fg_tram)
fg_tram.add_to(m)

fg_rail = folium.FeatureGroup(name="Linie kolejowe")
folium.GeoJson(
    train_railways,
    name="Kolej",
    style_function=lambda x: {
        "color": "black",
        "weight": 2
    }
).add_to(fg_rail)
fg_rail.add_to(m)

folium.GeoJson(
    lodzkie_proj,
    control=False,
    name="Województwo łódzkie",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)

fg_stops = folium.FeatureGroup(name="Stacje kolejowe")
for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(fg_stops)
fg_stops.add_to(m)

fg_walk = folium.FeatureGroup("Czas pieszo na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="walk_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojścia do stacji"
).add_to(fg_walk)
fg_walk.add_to(m)

fg_drive = folium.FeatureGroup("Czas autem na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="drive_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojazdu do stacji"
).add_to(fg_drive)
fg_drive.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
from folium.plugins import HeatMap

m = folium.Map(location=[51.76, 19.46], zoom_start=9)

fg_roads = folium.FeatureGroup(name="Drogi samochodowe", show=False)
folium.GeoJson(
    roads_car,
    name="Drogi",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1
    }
).add_to(fg_roads)
fg_roads.add_to(m)

fg_tram = folium.FeatureGroup(name="Linie tramwajowe", show=False)
folium.GeoJson(
    tram_railways,
    name="Tramwaje",
    style_function=lambda x: {
        "color": "brown",
        "weight": 1
    }
).add_to(fg_tram)
fg_tram.add_to(m)

fg_rail = folium.FeatureGroup(name="Linie kolejowe")
folium.GeoJson(
    train_railways,
    name="Kolej",
    style_function=lambda x: {
        "color": "black",
        "weight": 2
    }
).add_to(fg_rail)
fg_rail.add_to(m)

folium.GeoJson(
    lodzkie_proj,
    control=False,
    name="Województwo łódzkie",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)

fg_stops = folium.FeatureGroup(name="Stacje kolejowe")
for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(fg_stops)
fg_stops.add_to(m)

fg_walk = folium.FeatureGroup("Czas pieszo na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="walk_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojścia do stacji"
).add_to(fg_walk)
fg_walk.add_to(m)

fg_drive = folium.FeatureGroup("Czas autem na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="drive_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojazdu do stacji"
).add_to(fg_drive)
fg_drive.add_to(m)



gdf = gpd.read_file("GRID_NSP2021_RES.shp")
gdf['centroid'] = gdf.geometry.centroid

gdf_centroids = gdf.copy()
gdf_centroids['centroid'] = gdf_centroids['centroid'].to_crs(epsg=4326)

gdf_centroids['lat'] = gdf_centroids['centroid'].y
gdf_centroids['lon'] = gdf_centroids['centroid'].x

heat_data = [
    [row['lat'], row['lon'], row['RES']]
    for idx, row in gdf_centroids.iterrows()
    if lodzkie_proj.loc[11, 'geometry'].contains(Point(row['lon'], row['lat']))
]

fg_population = folium.FeatureGroup("Gęstosć zaludnienia", show=False)
HeatMap(heat_data, radius=7, blur=8, max_zoom=10).add_to(fg_population)
fg_population.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [ ]:
from folium.plugins import DualMap

mm = DualMap(
    location=[51.76, 19.46],
    zoom_start=8
)

def make_map(mode):
    m = folium.Map(location=[51.76, 19.46], zoom_start=8)

    if mode == 1:
        fg_roads = folium.FeatureGroup(name="Drogi samochodowe", show=False)
        folium.GeoJson(
            roads_car,
            name="Drogi",
            style_function=lambda x: {
                "color": "gray",
                "weight": 1
            }
        ).add_to(fg_roads)
        fg_roads.add_to(m)

        fg_tram = folium.FeatureGroup(name="Linie tramwajowe", show=False)
        folium.GeoJson(
            tram_railways,
            name="Tramwaje",
            style_function=lambda x: {
                "color": "brown",
                "weight": 1
            }
        ).add_to(fg_tram)
        fg_tram.add_to(m)

    fg_rail = folium.FeatureGroup(name="Linie kolejowe")
    folium.GeoJson(
        train_railways,
        name="Kolej",
        style_function=lambda x: {
            "color": "black",
            "weight": 2
        }
    ).add_to(fg_rail)
    fg_rail.add_to(m)

    folium.GeoJson(
        lodzkie_proj,
        control=False,
        name="Województwo łódzkie",
        style_function=lambda x: {
            "fillColor": "none",
            "color": "red",
            "weight": 2
        }
    ).add_to(m)

    if mode == 1:
        fg_stops = folium.FeatureGroup(name="Stacje kolejowe")
    else:
        fg_stops = folium.FeatureGroup(name="Stacje kolejowe", show=False)
    for _, row in train_stops.iterrows():
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            tooltip=row['name'],
            icon=folium.Icon(color="black", icon="train", prefix="fa")
        ).add_to(fg_stops)
    fg_stops.add_to(m)

    if mode == 1:
        fg_walk = folium.FeatureGroup("Czas pieszo na stacje", show=False)
        folium.raster_layers.ImageOverlay(
            image="walk_travel_time.png",
            bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
            opacity=0.6,
            name="Czas dojścia do stacji"
        ).add_to(fg_walk)
        fg_walk.add_to(m)

        fg_drive = folium.FeatureGroup("Czas autem na stacje")
        folium.raster_layers.ImageOverlay(
            image="drive_travel_time.png",
            bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
            opacity=0.6,
            name="Czas dojazdu do stacji"
        ).add_to(fg_drive)
        fg_drive.add_to(m)
    else:
        fg_population = folium.FeatureGroup("Gęstosć zaludnienia")
        HeatMap(heat_data, radius=7, blur=8, max_zoom=10).add_to(fg_population)
        fg_population.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    return m

map0 = make_map(0)
for child in map0._children.values():
    mm.m1.add_child(child)

map1 = make_map(1)
for child in map1._children.values():
    mm.m2.add_child(child)

mm

In [ ]:
rows, cols = np.where(time_raster_masked > np.nanpercentile(time_raster_masked, 70))
low_time_points = []
low_time_values = []

for r, c in zip(rows, cols):
    val = time_raster_masked[r, c]
    if np.isfinite(val):
        low_time_points.append(Point(xx[r, c], yy[r, c]))
        low_time_values.append(val)

low_time_gdf = gpd.GeoDataFrame({'time': low_time_values}, geometry=low_time_points, crs=2180)

heat_gdf = gpd.GeoDataFrame(
    heat_data,
    columns=['lat', 'lon', 'RES'],
    geometry=[Point(lon, lat) for lat, lon, _ in heat_data],
    crs="EPSG:4326"
)
high_heat_gdf = heat_gdf[heat_gdf['RES'] > np.percentile(heat_gdf['RES'], 90)]

low_time_gdf = low_time_gdf.to_crs(epsg=2180)
high_heat_gdf = high_heat_gdf.to_crs(epsg=2180)
low_time_high_heat = gpd.sjoin_nearest(
    low_time_gdf, high_heat_gdf,
    distance_col='dist', max_distance=1500
)

sorted_points = low_time_high_heat.sort_values('RES', ascending=False).copy()

selected_points = []
selected_indices = []

for idx, row in sorted_points.iterrows():
    point = row.geometry
    if all(point.distance(existing_point) > 8000 for existing_point in selected_points):
        selected_points.append(point)
        selected_indices.append(idx)
    if len(selected_points) >= 30:
        break

top_points = sorted_points.loc[selected_indices]

train_railways = train_railways.to_crs(epsg=2180)
top_points['near_rail'] = top_points.geometry.apply(
    lambda p: train_railways.distance(p).min() <= 5000
)



fig, ax = plt.subplots(figsize=(10,10))
lodzkie_proj = lodzkie_proj.to_crs(epsg=2180)
lodzkie_proj.plot(ax=ax, color='lightgrey')
top_points[~top_points['near_rail']].plot(
    ax=ax, facecolor='none', edgecolor='purple', linewidth=2, markersize=200
)

top_points[top_points['near_rail']].plot(
    ax=ax, facecolor='none', edgecolor='yellow', linewidth=2, markersize=200
)
plt.show()

In [ ]:
from folium import IFrame

train_railways = train_railways.to_crs(epsg=4326)
lodzkie_proj = lodzkie_proj.to_crs(epsg=4326)

m = folium.Map(location=[51.76, 19.46], zoom_start=8)

fg_roads = folium.FeatureGroup(name="Drogi samochodowe", show=False)
folium.GeoJson(
    roads_car,
    name="Drogi",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1
    }
).add_to(fg_roads)
fg_roads.add_to(m)

fg_tram = folium.FeatureGroup(name="Linie tramwajowe", show=False)
folium.GeoJson(
    tram_railways,
    name="Tramwaje",
    style_function=lambda x: {
        "color": "brown",
        "weight": 1
    }
).add_to(fg_tram)
fg_tram.add_to(m)

fg_rail = folium.FeatureGroup(name="Linie kolejowe")
folium.GeoJson(
    train_railways,
    name="Kolej",
    style_function=lambda x: {
        "color": "black",
        "weight": 2
    }
).add_to(fg_rail)
fg_rail.add_to(m)

folium.GeoJson(
    lodzkie_proj,
    control=False,
    name="Województwo łódzkie",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "red",
        "weight": 2
    }
).add_to(m)


fg_stops = folium.FeatureGroup(name="Stacje kolejowe")
for _, row in train_stops.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row['name'],
        icon=folium.Icon(color="black", icon="train", prefix="fa")
    ).add_to(fg_stops)
fg_stops.add_to(m)

fg_walk = folium.FeatureGroup("Czas pieszo na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="walk_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojścia do stacji"
).add_to(fg_walk)
fg_walk.add_to(m)

fg_drive = folium.FeatureGroup("Czas autem na stacje", show=False)
folium.raster_layers.ImageOverlay(
    image="drive_travel_time.png",
    bounds=[[lat_min, lon_min - 0.015], [lat_max+0.005, lon_max -0.02]],
    opacity=0.6,
    name="Czas dojazdu do stacji"
).add_to(fg_drive)
fg_drive.add_to(m)

fg_population = folium.FeatureGroup("Gęstosć zaludnienia", show=False)
HeatMap(heat_data, radius=7, blur=8, max_zoom=10).add_to(fg_population)
fg_population.add_to(m)

fg_top_points = folium.FeatureGroup(name="Top lokacje", show=True)
for _, row in top_points.to_crs(epsg=4326).iterrows():
    color = "rgb(120, 120, 0)" if row['near_rail'] else "purple"
    folium.Circle(
        location=[row.geometry.y, row.geometry.x],
        radius=2500,
        color=color,
        fill=False,
        weight=4
    ).add_to(fg_top_points)
fg_top_points.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

legend_html = """
<div style="
    background-color: white;
    border:2px solid grey;
    padding: 10px;
    font-size:14px;
    width: 180px;
    opacity: 0.8;
">
<b>Legenda</b><br>
<span style="background: yellow; width: 15px; height: 15px; display: inline-block; margin-right: 5px;"></span> Blisko torów<br>
<span style="background: purple; width: 15px; height: 15px; display: inline-block; margin-right: 5px;"></span> Daleko od torów
</div>
"""

iframe = IFrame(html=legend_html, width=200, height=100)
popup = folium.Popup(iframe, max_width=200)

folium.Marker(
    location=[52.15, 17.7],
    icon=folium.DivIcon(html=legend_html)
).add_to(m)

m